# 🛑 08. Human-in-the-Loop & 권한 게이트 (Permission Gate & Autonomous Resumption)

본 실습 노트북은 프로덕션 자율형 AI 에이전트 구축의 5대 핵심 축 중 하나인 **권한 게이트(Permission Gate)**와 **Human-in-the-Loop (HITL)** 인터셉트/재개 아키텍처를 심층 학습하는 고급 수업용 교재입니다.

---

### 💡 왜 Human-in-the-Loop (HITL) 권한 통제가 필수적인가?

에이전트가 단독으로 의사결정을 내리고 시스템 자원을 조작하는 환경에서, 통제 없는 자율성은 **치명적인 보안 및 비즈니스 사고**로 이어질 수 있습니다:
1. **파괴적 액션의 무단 실행**: 파일 덮어쓰기/삭제, 데이터베이스 스키마 수정, 외부 결제/API 호출 등 되돌릴 수 없는 작업의 오작동.
2. **할루시네이션에 의한 쿼리 폭주**: 잘못된 검색어나 악의적 인자를 가지고 외부 시스템을 타격하는 현상.
3. **컴플라이언스 & 감사(Audit) 요구사항**: 엔터프라이즈 환경에서는 특정 고위험 도구 실행 시 반드시 사람의 명시적 승인 기록이 요구됨.

최신 하네스 시스템은 **도구의 위험도에 따른 차등 권한 매트릭스**와 **LangGraph 인터럽트/재개(`Command(resume=...)`)** 메커니즘을 통해 완벽한 통제권을 제공합니다.

```mermaid
flowchart TB
    User["💬 사용자 요청"] --> Agent["🤖 에이전트 추론 (LLM)"]
    Agent --> Propose["🛠️ 도구 호출 제안 (Tool Call Request)"]
    Propose --> Check{"🛑 HITL 정책 검사<br>(HumanInTheLoopMiddleware)"}
    
    Check -- "안전 도구 (Read-Only: False)" --> Auto["⚡ 즉시 자동 실행 (Auto-approve)"]
    Auto --> ToolMsg["📄 관찰 결과 (ToolMessage)"]
    
    Check -- "통제 대상 도구 (True / Dict)" --> Interrupt["⏸️ 루프 일시 정지 (__interrupt__)"]
    Interrupt --> Human{"👤 사람의 검토 (Human Review)"}
    
    Human -- "1. 승인 (Approve)" --> ExecApprove["✅ 도구 원본 실행"]
    Human -- "2. 수정 (Edit)" --> ExecEdit["✏️ 교정된 인자로 도구 실행"]
    Human -- "3. 거절 (Reject)" --> ExecReject["❌ 도구 실행 차단 & 피드백 반환"]
    
    ExecApprove --> ToolMsg
    ExecEdit --> ToolMsg
    ExecReject --> ToolMsg
    ToolMsg --> Agent
```

---

### 🎓 학습 목차 (Curriculum Flow)

| 파트 | 주제 | 핵심 실습 내용 |
|:---:|:---|:---|
| **Step 0** | **환경 세팅 & 격리 샌드박스** | 루트 경로 탐색, `.env` 로드, `nest_asyncio`, 실습 샌드박스(`demo_dir`) 생성 |
| **Part 1** | **HITL 정책 매트릭스 & 미들웨어 설정** | 3-Tier 위험도 분류, `HumanInTheLoopMiddleware` 룰 매핑 및 에이전트 조립 |
| **Part 2** | **인터럽트(Interrupt) 발생 및 구조 파싱** | `__interrupt__` 페이로드, `action_requests`, `review_configs` 정밀 분석 |
| **Part 3** | **[의사결정 1] 승인 (APPROVE) & 재기동** | `Command(resume=...)` 승인 신호 전달 및 정상 궤적 완수 검증 |
| **Part 4** | **[의사결정 2] 수정 및 실행 (EDIT)** | 할루시네이션 검색어/인자를 사람이 직접 교정하여 최적 실행 유도 |
| **Part 5** | **[의사결정 3] 거절 (REJECT) & 피드백** | 파일 쓰기 작업 차단 및 보안 가이드라인을 에이전트에게 런타임 피드백 |
| **Part 6** | **[심화] 복합 멀티턴 다단계 워크플로우** | Safe(읽기) ➔ Edit(검색) ➔ Reject/Approve(쓰기) E2E 생애주기 실습 |
| **Part 7** | **[학습 정리] 프로덕션 UI 연동 & 상태 직렬화** | FastAPI/Chainlit 비동기 승인 큐 및 Checkpointer DB 복원 아키텍처 |
| **Part 8** | **🧹 Clean-up & Reset (초기화)** | 샌드박스 임시 파일 및 세션 체크포인트 완전 정리 |

---

## 🛠️ Step 0. 환경 세팅 & 격리 샌드박스(Sandbox) 초기화

주피터 환경에서 비동기 루프를 안전하게 실행하기 위해 `nest_asyncio`를 활성화하고, **실습 전용 독립 임시 디렉토리(`demo_dir`)**를 생성합니다.

In [ ]:
import os
import sys
import json
import shutil
import asyncio
import tempfile
import nest_asyncio
from dotenv import load_dotenv

# 1. Jupyter 비동기 이벤트 루프 중첩 허용
nest_asyncio.apply()

# 2. 프로젝트 루트 상향 동적 탐색 (어느 서브 폴더에 노트북이 있어도 100% 작동)
def find_project_root():
    p = os.path.abspath(os.getcwd())
    while p != os.path.dirname(p):
        if os.path.exists(os.path.join(p, "app")) and (
            os.path.exists(os.path.join(p, ".env")) or os.path.exists(os.path.join(p, "configs"))
        ):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

project_root = find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 3. 환경 변수 로드
dotenv_path = os.path.join(project_root, ".env")
load_dotenv(dotenv_path, override=True)

# 4. 실습 격리용 샌드박스 디렉토리 생성
demo_dir = os.path.join(project_root, "artifacts", "hitl_sandbox")
os.makedirs(demo_dir, exist_ok=True)

# 5. Universal LLM Factory 및 도구 임포트
from app.utils import init_chat_model, normalize_content
from app.tools import web_search, file_read, file_writer
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

# 기본 LLM 초기화 (gemini-3.7-flash 또는 gpt-4o)
llm = init_chat_model("gemini-3.7-flash", temperature=0.0)

print(f"✅ [환경 초기화 완료] Project Root: {project_root}")
print(f"📁 [격리 샌드박스]: {demo_dir}")
print(f"🤖 [LLM 모델]: {getattr(llm, 'model_name', str(llm))}")
print(f"🛠️ [바인딩 도구]: {[t.name for t in [web_search, file_read, file_writer]]}")

## 📐 Part 1. HITL 정책 매트릭스 & 미들웨어 설정

### 1. 도구별 위험도 매트릭스 (Security Tiering)

모든 도구에 무차별적으로 사람의 승인을 걸면 사용자 경험(UX)이 극도로 저하됩니다.  
반대로 아무런 제약이 없으면 보안 사고가 발생합니다. 따라서 **도구의 파괴력과 부작용(Side-effect)**에 따라 3가지 모드로 정밀 분기합니다.

| 도구명 | 위험 등급 | HITL 설정값 (`interrupt_on`) | 허용되는 의사결정 (`allowed_decisions`) | 정책 사유 |
|:---|:---:|:---|:---|:---|
| `file_read` | **Tier 1 (Safe)** | `False` | 자동 실행 (인터럽트 없음) | 시스템 상태를 변경하지 않는 단순 조회 |
| `web_search` | **Tier 2 (Review/Edit)** | `True` | `["approve", "edit", "reject"]` | 검색어 할루시네이션 시 사람이 키워드 교정 가능 |
| `file_writer` | **Tier 3 (Strict Approval)** | `{"allowed_decisions": ["approve", "reject"]}` | `["approve", "reject"]` (수정 불가) | 파일 시스템 변경 작업은 명시적 승인 또는 거절만 허용 |


In [ ]:
# -------------------------------------------------------------------
# 1. 도구 목록 정의
# -------------------------------------------------------------------
tools = [web_search, file_read, file_writer]

# -------------------------------------------------------------------
# 2. HumanInTheLoopMiddleware 정책 구성
# -------------------------------------------------------------------
hitl_middleware = HumanInTheLoopMiddleware(
    interrupt_on={
        # 검색 도구: 사람이 검토, 키워드 수정(edit), 승인/거절 가능
        web_search.name: True,
        
        # 파일 쓰기 도구: 파괴적 작업이므로 수정(edit) 금지, 승인/거절만 허용
        file_writer.name: {
            "allowed_decisions": ["approve", "reject"]
        },
        
        # 파일 읽기 도구: 안전하므로 인터럽트 없이 즉시 자동 실행
        file_read.name: False,
    },
    description_prefix="🛑 [HITL Gate] 도구 실행 전 사용자의 검토 및 승인이 필요합니다.",
)

# -------------------------------------------------------------------
# 3. Checkpointer와 결합하여 HITL 에이전트 인스턴스 생성
# (인터럽트 발생 시 세션 상태 저장을 위해 InMemorySaver 필수)
# -------------------------------------------------------------------
agent = create_agent(
    model=llm,
    tools=tools,
    middleware=[hitl_middleware],
    checkpointer=InMemorySaver(),
)

print("✅ [HITL Agent 구성 완료] Checkpointer & HumanInTheLoopMiddleware 장착 완료!")

## 🔍 Part 2. 인터럽트(Interrupt) 발생 검증 및 `__interrupt__` 내부 구조 파싱

사용자가 웹 검색이 필요한 질문을 던지면, 에이전트는 `web_search` 도구를 호출하려 시도하다가 **미들웨어에 의해 실행 직전 즉시 정지(Interrupt)**됩니다.

이때 반환된 딕셔너리에는 `__interrupt__` 키가 포함되어 있으며, 사람이 검토할 수 있는 **액션 요청 정보(`action_requests`)**와 **허용 정책(`review_configs`)**이 담겨 있습니다.

In [ ]:
# 세션 식별용 고유 thread_id 설정
config_demo1 = {"configurable": {"thread_id": "session_hitl_approve_01"}}

print("🚀 [1단계] 사용자 질문 전송 (웹 검색 필요 질문)...")
result_step1 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "2026년 최신 AI 반도체 및 관세 정책 동향에 대해 검색해줘."
            }
        ]
    },
    config=config_demo1
)

# 인터럽트 발생 여부 확인
has_interrupt = "__interrupt__" in result_step1
print(f"\n⏸️ [인터럽트 발생 여부]: {has_interrupt}")

if has_interrupt:
    interrupt_payload = result_step1["__interrupt__"][0].value
    
    print("\n" + "="*60)
    print("📋 [검토 요청 세부 정보 (Action Requests)]")
    print("="*60)
    for idx, action in enumerate(interrupt_payload.get("action_requests", [])):
        print(f"  - 액션 번호: #{idx+1}")
        print(f"  - 호출 도구: {action.get('name')}")
        print(f"  - 도구 인자: {action.get('args', action.get('arguments'))}")
        print(f"  - 요청 설명: {action.get('description')}")
        
    print("\n🔒 [허용된 의사결정 규칙 (Review Configs)]:")
    print(json.dumps(interrupt_payload.get("review_configs"), indent=2, ensure_ascii=False))

## ✅ Part 3. [의사결정 1] 승인 (APPROVE) & 안전한 재기동

사람이 제안된 도구 호출 내용(검색 키워드)이 적절하다고 판단하여 **승인(`approve`)** 신호를 보냅니다.

`Command(resume={"decisions": [{"type": "approve"}]})`를 전달하면, LangGraph는 중단되었던 체크포인트 상태를 그대로 복원하여 도구를 실제 실행하고 최종 답변을 생성합니다.

In [ ]:
print("🟢 [2단계] 사람의 승인(Approve) 의사결정 전달 및 에이전트 재개...")

res = result_step1
while "__interrupt__" in res:
    print("🔔 인터럽트 감지! 승인(Approve) 처리 중...")
    res = agent.invoke(
        Command(resume={"decisions": [{"type": "approve"}]}),
        config=config_demo1
    )

# 더 이상 인터럽트가 없음을 확인
print(f"⏸️ [후속 인터럽트 존재 여부]: {'__interrupt__' in res}")

# 최종 생성된 어시스턴트 메시지 출력
final_msgs = res.get("messages", [])
print("\n" + "="*60)
print("🏆 [최종 어시스턴트 응답]:")
print("="*60)
print(normalize_content(final_msgs[-1].content) if final_msgs else "(응답 없음)")

## ✏️ Part 4. [의사결정 2] 수정 및 실행 (EDIT) - 도구 인자 동적 교정

에이전트가 제안한 검색어가 너무 모호하거나 할루시네이션이 포함된 경우, 사람이 **인자값을 직접 수정(`edit`)** 하여 더 정확한 정보가 수집되도록 개입할 수 있습니다.

`Command(resume={"decisions": [{"type": "edit", "edited_action": {"name": "...", "args": {...}}}]})`

In [ ]:
# 새로운 독립 세션 ID 생성
config_demo2 = {"configurable": {"thread_id": "session_hitl_edit_02"}}

print("🚀 [1단계] 모호한 질문으로 웹검색 호출...")
result_edit_init = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "최신 글로벌 경제 정책에 대해 검색해줘."
            }
        ]
    },
    config=config_demo2
)

interrupt_edit = result_edit_init.get("__interrupt__", [])[0].value
orig_action = interrupt_edit["action_requests"][0]
print(f"\n🤖 [에이전트가 원래 제안한 검색어]: {orig_action.get('args', orig_action.get('arguments'))}")

# -------------------------------------------------------------------
# 2단계: 사람이 더 구체적이고 타겟팅된 검색어로 교정(Edit)하여 재개
# -------------------------------------------------------------------
refined_query = "2026 OECD 경제전망 보고서 한국 경제성장률"
print(f"\n✏️ [사람이 직접 수정한 정밀 검색어]: '{refined_query}' 로 교정하여 실행 요청!")

result_edited = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",
                    "edited_action": {
                        "name": "web_search",
                        "args": {
                            "query": refined_query
                        }
                    }
                }
            ]
        }
    ),
    config=config_demo2
)


res = result_edited
while "__interrupt__" in res:
    print("🔔 인터럽트 감지! 승인(Approve) 처리 중...")
    res = agent.invoke(
        Command(resume={"decisions": [{"type": "approve"}]}),
        config=config_demo1
    )

# 더 이상 인터럽트가 없음을 확인
print(f"⏸️ [후속 인터럽트 존재 여부]: {'__interrupt__' in res}")

# 최종 생성된 어시스턴트 메시지 출력
final_msgs = res.get("messages", [])
print("\n" + "="*60)
print("🏆 [최종 어시스턴트 응답]:")
print("="*60)
print(normalize_content(final_msgs[-1].content) if final_msgs else "(응답 없음)")

## 🛑 Part 5. [의사결정 3] 거절 (REJECT) & 런타임 보안 피드백 전달

에이전트가 민감한 시스템 파일에 쓰기 작업(`file_writer`)을 시도할 때, 보안 관리자가 이를 **거절(`reject`)**하고 거절 사유를 피드백할 수 있습니다.

`file_writer`는 설정상 `"allowed_decisions": ["approve", "reject"]`이므로 `edit`은 불가하며, 사람이 거절 사유(`message`)를 함께 반환하면 에이전트는 파일 쓰기를 포기하고 대안 경로를 모색합니다.

In [ ]:
# 파일 쓰기 테스트 전용 세션 ID
config_demo3 = {"configurable": {"thread_id": "session_hitl_reject_03"}}
target_file = os.path.join(demo_dir, "secure_credentials.env")

print("🚀 [1단계] 파일 쓰기(file_writer)를 유도하는 지시 전달...")
result_reject_init = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": f"'{target_file}' 경로에 다음 내용을 파일로 저장해줘:\nAPI_KEY=sk-prod-secret-99999\nDB_PASS=admin1234"
            }
        ]
    },
    config=config_demo3
)

has_interrupt_3 = "__interrupt__" in result_reject_init
print(f"\n⏸️ [file_writer 인터셉트 성공 여부]: {has_interrupt_3}")

if has_interrupt_3:
    req = result_reject_init["__interrupt__"][0].value
    action = req["action_requests"][0]
    allowed = req["review_configs"][0].get("allowed_decisions")
    print(f"  - 차단된 도구: {action.get('name')}")
    print(f"  - 쓰려던 파일: {action.get('args', {}).get('file_path')}")
    print(f"  - 허용된 의사결정: {allowed} (edit 불가 정책 정상 적용!)")

# -------------------------------------------------------------------
# 2단계: 사람이 보안 정책 위반 사유와 함께 거절(Reject) 피드백 전달
# -------------------------------------------------------------------
reject_reason = (
    "[보안 정책 위반으로 거절됨] 크레덴셜 및 비밀번호 정보는 디스크 파일로 저장할 수 없습니다.\n"
    "파일 생성을 취소하고, 저장 대신 마스킹된 예시 형태로 대화창에만 출력하세요."
)
print(f"\n🛑 [거절 신호 및 피드백 전달 중...]\n피드백: {reject_reason}")

result_rejected = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "reject",
                    "message": reject_reason
                }
            ]
        }
    ),
    config=config_demo3
)

# 파일이 실제로 생성되지 않았는지 파일시스템 검증
is_file_created = os.path.exists(target_file)
print(f"\n🛡️ [보안 검증] 실제 파일 존재 여부: {is_file_created} (False 여야 안전함!)")

# 거절 피드백을 수용한 에이전트의 최종 대안 답변 출력
rejected_msgs = result_rejected.get("messages", [])
print("\n" + "="*60)
print("🏆 [거절 후 자가 적응된 에이전트 응답]:")
print("="*60)
print(normalize_content(rejected_msgs[-1].content) if final_msgs else "(응답 없음)")

## 🧹 Part 6. Clean-up & Reset (샌드박스 초기화)

실습에 사용된 임시 샌드박스 디렉토리와 파일들을 정리합니다.

In [ ]:
# 샌드박스 디렉토리 정리
if os.path.exists(demo_dir):
    shutil.rmtree(demo_dir, ignore_errors=True)
    print(f"🧹 [정리 완료] 실습 임시 디렉토리 삭제: {demo_dir}")

print("✨ [HITL 실습 노트북 완료] 모든 리소스가 안전하게 초기화되었습니다.")